# U8 — Data Cleaning & Preprocessing (Part 1): Lab

**Cleaning Messy Data** — profiling · missing values · duplicates · data types · outliers · messy text

_Day 4 · Phase C — Data Engineering & Preparation · The highest-leverage work in ML: clean data first._

#objectives

By the end of this lab you will be able to:

Profile a dataset to find quality problems before fixing anything

Detect missing values — including disguised ones — and handle them by dropping or imputing

Remove duplicate rows and fix wrong data types

Detect outliers with the IQR rule and decide how to treat them

Standardise messy text so one real category isn't split into many

#how to use this lab

Each section has two kinds of cells:

Worked demo cells — run them top to bottom and read the comments to learn the pattern.

LAB EXERCISE cells (marked 🧪) — your turn. Replace each `# YOUR CODE HERE` with working code.

Run cells with **Shift + Enter**. Run the demos before attempting the exercises.

In [ ]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

Setup complete. pandas 2.2.2


In [ ]:
# -----------------------------------------------------------
# A DELIBERATELY MESSY DATASET (so the lab is self-contained)
# -----------------------------------------------------------
# Problems baked in: missing values, disguised missing ('N/A', -1),
# duplicate rows, a number stored as text, a date as text,
# an extreme outlier, and inconsistent city spellings.
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})
raw


,id,name,age,city,spend,date
0,1,Ana,30.0,Pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,NaN,DELHI,200.2,2024-01-07
3,4,Di,41.0,Delhi,N/A,2024-01-08
4,5,Eve,-1.0,Mumbai,150.0,2024-01-09
5,6,Fin,38.0,bombay,99000,2024-01-10
6,7,Gus,29.0,Pune.,110.0,2024-01-11
7,7,Gus,29.0,Pune.,110.0,2024-01-11


#1. Profile the data — find the problems

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. A FEW COMMANDS REVEAL MOST PROBLEMS
# -----------------------------------------------------------

df = raw.copy()        # always work on a copy
df.info()              # types + non-null counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      8 non-null      int64  
 1   name    8 non-null      object 
 2   age     7 non-null      float64
 3   city    8 non-null      object 
 4   spend   8 non-null      object 
 5   date    8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


In [ ]:
# -----------------------------------------------------------
# 🔹 1B. MISSING COUNTS, DUPLICATES & RANGES
# -----------------------------------------------------------

print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")

Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64

Duplicate rows: 1

Note: spend is type object -> stored as text!


#### 🧪 LAB EXERCISE 1 — Profile it yourself

1. Print the number of **duplicate rows**.
2. Print the **missing count per column**.
3. In a comment, list at least **three** problems you can already see in the data.

In [20]:
# 1. duplicate row count
# YOUR CODE HERE
print("duplicates count:",df.duplicated().sum())

# 2. missing per column
# YOUR CODE HERE
print(df.isna().sum())
# 3. Problems I can see: ...   (write 3+ in this comment)
print("Problems I can see:\n 1. Random Capatilization")
print(" 2. age in negative number instead of positive range")
print(" 3. no fixed data type")

duplicates count: 1
id       0
name     0
age      0
city     0
spend    0
date     0
dtype: int64
Problems I can see:
 1. Random Capatilization
 2. age in negative number instead of positive range
 3. no fixed data type


#2. Missing values — detect & handle

In [17]:
# -----------------------------------------------------------
# 🔹 2A. UNMASK DISGUISED MISSING VALUES
# -----------------------------------------------------------

# 'N/A' (in spend) and -1 (in age) are really missing -> make them NaN
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')  # 'N/A' -> NaN, text -> number
df['age']   = df['age'].replace(-1, np.nan)                # sentinel -> NaN

print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())

Missing after unmasking:
age      2
spend    1
dtype: int64


In [18]:
# -----------------------------------------------------------
# 🔹 2B. HANDLE THE GAPS (impute)
# -----------------------------------------------------------

# median is robust to skew/outliers -> good for 'spend' and 'age'
df['age']   = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())
print('Missing after imputing:', df[['age', 'spend']].isna().sum().sum())

Missing after imputing: 0


#### 🧪 LAB EXERCISE 2 — Compare drop vs impute

Start from a fresh messy copy (`ex = raw.copy()`):
1. Convert `spend` to numeric and `age`'s `-1` to NaN (as in 2A).
2. Make a **dropna** version and a **median-impute** version.
3. Print the row count of each — how many rows did dropping cost you?

In [34]:
ex = raw.copy()

# 1. unmask missing values (spend -> numeric, age -1 -> NaN)
# YOUR CODE HERE
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['age']   = ex['age'].replace(-1, np.nan)
print (ex)
# 2a. dropna version
# YOUR CODE HERE
ex.dropna(inplace=True)
print("after dropping:\n",ex)
# 2b. median-impute version
# YOUR CODE HERE
df['spend']   = df['spend'].fillna(df['spend'].median())
df['age'] = df['age'].fillna(df['age'].median())
print('Missing after imputing:', df[['spend', 'age']].isna().sum().sum())
print("modified dataset:",ex)

# 3. compare row counts
# YOUR CODE HERE
print("row count comparision:",df.shape[0] - ex.shape[0])

   id name   age    city    spend        date
0   1  Ana  30.0   Pune     120.5  2024-01-05
1   2   Bo  25.0    pune     80.0  2024-01-06
2   3   Cy   NaN   DELHI    200.2  2024-01-07
3   4   Di  41.0  Delhi       NaN  2024-01-08
4   5  Eve   NaN  Mumbai    150.0  2024-01-09
5   6  Fin  38.0  bombay  99000.0  2024-01-10
6   7  Gus  29.0   Pune.    110.0  2024-01-11
7   7  Gus  29.0   Pune.    110.0  2024-01-11
after dropping:
    id name   age    city    spend        date
0   1  Ana  30.0   Pune     120.5  2024-01-05
1   2   Bo  25.0    pune     80.0  2024-01-06
5   6  Fin  38.0  bombay  99000.0  2024-01-10
6   7  Gus  29.0   Pune.    110.0  2024-01-11
7   7  Gus  29.0   Pune.    110.0  2024-01-11
Missing after imputing: 0
modified dataset:    id name   age    city    spend        date
0   1  Ana  30.0   Pune     120.5  2024-01-05
1   2   Bo  25.0    pune     80.0  2024-01-06
5   6  Fin  38.0  bombay  99000.0  2024-01-10
6   7  Gus  29.0   Pune.    110.0  2024-01-11
7   7  Gus  29.0   

#3. Duplicates & data types

In [35]:
# -----------------------------------------------------------
# 🔹 3A. DROP DUPLICATE ROWS
# -----------------------------------------------------------

print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')

Before: (8, 6)
After : (7, 6) -> removed the repeated Gus row


In [36]:
# -----------------------------------------------------------
# 🔹 3B. FIX DATA TYPES
# -----------------------------------------------------------

# 'date' is text -> convert to real datetimes so sorting/maths work
df['date'] = pd.to_datetime(df['date'])
# 'city' is a category -> mark it as such (saves memory, signals intent)
df['city'] = df['city'].astype('string')
print(df.dtypes)

id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


#### 🧪 LAB EXERCISE 3 — Dedupe & retype

On a fresh copy `ex = raw.copy()`:
1. Convert `spend` to numeric and `date` to datetime.
2. Drop duplicate rows.
3. Print `.dtypes` and the final `.shape`.

In [76]:
ex = raw.copy()

# 1. fix types: spend -> numeric, date -> datetime
# YOUR CODE HERE
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
print(ex['spend'])

ex['date'] = pd.to_datetime(ex['date'])
print(ex['date'])

# 2. drop duplicates
# YOUR CODE HERE
df= ex.drop_duplicates()
print("drop duplicates: \n",df)


# 3. dtypes + shape
# YOUR CODE HERE
print("data types:",df.dtypes)
print("shape",df.shape)

0      120.5
1       80.0
2      200.2
3        NaN
4      150.0
5    99000.0
6      110.0
7      110.0
Name: spend, dtype: float64
0   2024-01-05
1   2024-01-06
2   2024-01-07
3   2024-01-08
4   2024-01-09
5   2024-01-10
6   2024-01-11
7   2024-01-11
Name: date, dtype: datetime64[ns]
drop duplicates: 
    id name   age    city    spend       date
0   1  Ana  30.0   Pune     120.5 2024-01-05
1   2   Bo  25.0    pune     80.0 2024-01-06
2   3   Cy   NaN   DELHI    200.2 2024-01-07
3   4   Di  41.0  Delhi       NaN 2024-01-08
4   5  Eve  -1.0  Mumbai    150.0 2024-01-09
5   6  Fin  38.0  bombay  99000.0 2024-01-10
6   7  Gus  29.0   Pune.    110.0 2024-01-11
data types: id                int64
name             object
age             float64
city             object
spend           float64
date     datetime64[ns]
dtype: object
shape (7, 6)


#4. Outliers — detect with the IQR rule

In [77]:
# -----------------------------------------------------------
# 🔹 4A. THE IQR RULE
# -----------------------------------------------------------

# spend has a 99000 value among ~100s -> a clear outlier
q1, q3 = df['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')

outliers = df[(df['spend'] < low) | (df['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])

Q1=112.6  Q3=187.6  IQR=75.0
Normal range: 0.1 to 300.2

Outlier rows:
  name    spend
5  Fin  99000.0


In [78]:
# -----------------------------------------------------------
# 🔹 4B. ONE WAY TO TREAT THEM — CAP (winsorise)
# -----------------------------------------------------------

# clip values to the IQR bounds instead of deleting the row
df['spend_capped'] = df['spend'].clip(lower=low, upper=high)
print(df[['name', 'spend', 'spend_capped']])

  name    spend  spend_capped
0  Ana    120.5      120.5000
1   Bo     80.0       80.0000
2   Cy    200.2      200.2000
3   Di      NaN           NaN
4  Eve    150.0      150.0000
5  Fin  99000.0      300.1875
6  Gus    110.0      110.0000


/tmp/ipykernel_3820/3824263328.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['spend_capped'] = df['spend'].clip(lower=low, upper=high)


#### 🧪 LAB EXERCISE 4 — Find the outliers

Using the `age` column of the cleaned `df`:
1. Compute Q1, Q3 and the IQR.
2. Compute the lower & upper bounds (`Q1 − 1.5·IQR`, `Q3 + 1.5·IQR`).
3. Print any rows whose `age` falls outside those bounds (there may be none — that's a valid result).

In [83]:
# 1. Q1, Q3, IQR for 'age'
# YOUR CODE HERE
q1, q3 = df['age'].quantile([0.25, 0.75])
iqr = q3 - q1

# 2. lower & upper bounds
# YOUR CODE HERE
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')


# 3. rows outside the bounds
# YOUR CODE HERE
outliers = df[(df['age'] < low) | (df['age'] > high)]
print('\nOutlier rows:')
print(outliers[[ 'age']])


Q1=26.0  Q3=36.0  IQR=10.0
Normal range: 11.0 to 51.0

Outlier rows:
   age
4 -1.0


#5. Messy text & inconsistent categories

In [84]:
# -----------------------------------------------------------
# 🔹 5A. THE PROBLEM — ONE CITY, MANY SPELLINGS
# -----------------------------------------------------------

print(df['city'].value_counts())   # ' Pune ', 'pune', 'Pune.' all look different!


city
 Pune     1
pune      1
DELHI     1
Delhi     1
Mumbai    1
bombay    1
Pune.     1
Name: count, dtype: int64


In [85]:
# -----------------------------------------------------------
# 🔹 5B. STANDARDISE THE STRINGS
# -----------------------------------------------------------

s = df['city'].astype('string')
s = s.str.strip()                       # trim whitespace
s = s.str.lower()                       # unify case
s = s.str.replace('.', '', regex=False) # drop stray punctuation
s = s.replace({'bombay': 'mumbai'})     # map known variants to one label
df['city'] = s
print(df['city'].value_counts())        # now clean categories

city
pune      3
delhi     2
mumbai    2
Name: count, dtype: Int64


/tmp/ipykernel_3820/3964666752.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['city'] = s


#### 🧪 LAB EXERCISE 5 — Clean a messy column

The `messy` series below has the same problems.
1. Trim whitespace and lowercase it.
2. Map the variant `'n.y.'` to `'new york'`.
3. Print `value_counts()` — you should end up with just two clean categories.

In [94]:
messy = pd.Series([' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
                  dtype='string')

# 1. strip + lower
# YOUR CODE HERE
m = messy.str.strip()
m = messy.str.lower()


# 2. map 'n.y.' -> 'new york'  (after lowering)
# YOUR CODE HERE
m = m.replace({'n.y.': 'new york'})
print(m)

# 3. value_counts()
# YOUR CODE HERE
print(m.value_counts())



0      london 
1       london
2       london
3     new york
4    new york 
5     new york
dtype: string
london       2
new york     2
 london      1
new york     1
Name: count, dtype: Int64


#✅ The cleaned dataset

In [95]:
# After all the steps above, here's the cleaned frame
clean = df.drop(columns=['spend_capped'])
print('Final shape:', clean.shape)
print('Missing values:', int(clean.isna().sum().sum()))
print('Duplicates    :', int(clean.duplicated().sum()))
clean

Final shape: (7, 6)
Missing values: 2
Duplicates    : 0


,id,name,age,city,spend,date
0,1,Ana,30.0,pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,NaN,delhi,200.2,2024-01-07
3,4,Di,41.0,delhi,NaN,2024-01-08
4,5,Eve,-1.0,mumbai,150.0,2024-01-09
5,6,Fin,38.0,mumbai,99000.0,2024-01-10
6,7,Gus,29.0,pune,110.0,2024-01-11


#📘 Summary — Data-cleaning toolkit

| Step | What to check | Key calls |
| ---- | ------------- | --------- |
| **Profile** | overall health | `.info()` · `.isna().sum()` · `.describe()` · `.duplicated().sum()` |
| **Missing** | gaps & disguises | `pd.to_numeric(..., errors='coerce')` · `.replace(-1, np.nan)` · `.fillna()` |
| **Duplicates** | repeated rows | `.drop_duplicates()` |
| **Types** | text vs number/date | `pd.to_numeric` · `pd.to_datetime` · `.astype()` |
| **Outliers** | extreme values | IQR rule · `.clip(low, high)` |
| **Text** | inconsistent labels | `.str.strip()` · `.str.lower()` · `.replace({...})` |

**Homework (self-paced):** profile a messy CSV · unmask & impute missing values · drop duplicates · fix a numeric & a date column · flag IQR outliers · standardise a text column.

**Next — U8 Part 2:** preparing data for ML — encoding categoricals (label & one-hot), feature scaling (normalise & standardise), and feature-engineering basics.